# 08. RAG Pipeline Evaluation

검색 결과(SBERT+FAISS)와 Ollama LLM을 결합한 RAG 파이프라인을 평가합니다.
Zero-shot(검색 없이) vs RAG(검색+생성) 비교를 통해 RAG의 효과를 측정합니다.

| Item | Detail |
|------|--------|
| Task | RAG 답변 생성 + 품질 평가 |
| LLM | Ollama (자동 탐지) |
| Retriever | SBERT + FAISS (07에서 구축) |
| Metrics | ROUGE-L, BERTScore |
| Comparison | Zero-shot vs RAG |
| Environment | Local CPU (Ollama 필요) |

---
## 1. Environment Setup

In [ ]:
%%capture
!pip install -q rouge-score bert-score sentence-transformers faiss-cpu plotly httpx kaleido

In [ ]:
import os, json, time, warnings, pickle, re
import numpy as np
import pandas as pd
import httpx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'
from IPython.display import display, HTML
warnings.filterwarnings('ignore')

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/civilcomplaint-processed'
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    DATA_DIR = '../data/processed'
    OUT_DIR = '..'
    IS_KAGGLE = False

RESULTS_DIR = os.path.join(OUT_DIR, 'results')
INDEX_DIR = os.path.join(OUT_DIR, 'data/vectordb/faiss_index')
MODELS_DIR = os.path.join(OUT_DIR, 'models/embedding')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load previous retrieval results
bm25_results = None
sbert_results = None

bm25_path = os.path.join(RESULTS_DIR, 'retrieval_bm25_results.json')
sbert_path = os.path.join(RESULTS_DIR, 'retrieval_sbert_results.json')

if os.path.exists(bm25_path):
    with open(bm25_path) as f:
        bm25_results = json.load(f)
    print(f"BM25 results loaded: Recall@5={bm25_results.get('recall_at_5', 'N/A')}")

if os.path.exists(sbert_path):
    with open(sbert_path) as f:
        sbert_results = json.load(f)
    print(f"SBERT results loaded: Recall@5={sbert_results.get('recall_at_5', 'N/A')}")

OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
print(f"\nEnvironment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Ollama URL: {OLLAMA_BASE_URL}")

In [ ]:
# --- Ollama connection & available models ---
def check_ollama():
    """Check Ollama connectivity and list available models."""
    try:
        r = httpx.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=10)
        r.raise_for_status()
        models = r.json().get('models', [])
        model_names = [m['name'] for m in models]
        print(f"Ollama connected: {len(model_names)} models available")
        for m in models:
            size_gb = m.get('size', 0) / 1e9
            print(f"  - {m['name']} ({size_gb:.1f} GB)")
        return model_names
    except Exception as e:
        print(f"Ollama not available: {e}")
        print("Please start Ollama: `ollama serve`")
        return []

available_models = check_ollama()

# Auto-select model: prefer Korean-capable models
MODEL_PRIORITY = ['qwen2.5:7b', 'qwen2.5:latest', 'llama3.1:8b', 'gemma2:9b']
OLLAMA_MODEL = None
for mp in MODEL_PRIORITY:
    for am in available_models:
        if mp in am or am.startswith(mp.split(':')[0]):
            OLLAMA_MODEL = am
            break
    if OLLAMA_MODEL:
        break

if not OLLAMA_MODEL and available_models:
    OLLAMA_MODEL = available_models[0]

print(f"\nSelected model: {OLLAMA_MODEL}")

---
## 2. Data & Index Loading

In [ ]:
# --- Load QA data ---
qa_path = os.path.join(DATA_DIR, 'qa_pairs.parquet')
qa_df = pd.read_parquet(qa_path)
print(f"Total QA pairs: {len(qa_df):,}")

# Eval set: 1K sample (or less if data is small)
N_EVAL = min(1000, len(qa_df))
np.random.seed(42)
eval_indices = np.random.choice(len(qa_df), size=N_EVAL, replace=False)
eval_df = qa_df.iloc[eval_indices].reset_index(drop=True)

print(f"Eval set: {len(eval_df):,} samples")
print(f"Columns: {list(eval_df.columns)}")
eval_df.head(3)

In [ ]:
# --- Load FAISS index + embedding model ---
from sentence_transformers import SentenceTransformer
import faiss

# Determine best model from SBERT results
if sbert_results:
    best_model_name = sbert_results.get('best_model', 'ko-sbert-nli')
else:
    best_model_name = 'ko-sbert-nli'

model_path = os.path.join(MODELS_DIR, best_model_name)
if os.path.exists(model_path):
    embed_model = SentenceTransformer(model_path)
    print(f"Loaded local model: {model_path}")
else:
    hf_id = 'jhgan/ko-sbert-nli' if 'sbert' in best_model_name else f'BM-K/{best_model_name}'
    embed_model = SentenceTransformer(hf_id)
    print(f"Loaded HuggingFace model: {hf_id}")

# FAISS index
index_path = os.path.join(INDEX_DIR, 'index.faiss')
meta_path = os.path.join(INDEX_DIR, 'doc_metadata.pkl')

if os.path.exists(index_path):
    faiss_index = faiss.read_index(index_path)
    print(f"FAISS index loaded: {faiss_index.ntotal:,} vectors")
else:
    print("FAISS index not found - will build from scratch")
    corpus_questions = qa_df['question'].tolist()
    emb = embed_model.encode(corpus_questions, batch_size=256, show_progress_bar=True,
                              normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    faiss_index = faiss.IndexFlatIP(emb.shape[1])
    faiss_index.add(emb)
    print(f"Built FAISS index: {faiss_index.ntotal:,} vectors")

if os.path.exists(meta_path):
    with open(meta_path, 'rb') as f:
        doc_metadata = pickle.load(f)
    corpus_questions = doc_metadata['corpus_questions']
    corpus_answers = doc_metadata['corpus_answers']
    print(f"Doc metadata loaded: {len(corpus_answers):,} docs")
else:
    corpus_questions = qa_df['question'].tolist()
    corpus_answers = qa_df['answer'].tolist()
    print(f"Using full QA as corpus: {len(corpus_answers):,} docs")

print(f"\nEmbedding dim: {embed_model.get_sentence_embedding_dimension()}")
print(f"Index vectors: {faiss_index.ntotal:,}")

---
## 3. RAG Prompt Template

In [ ]:
# --- Prompt templates (matches backend/app/services/generate_service.py) ---

RAG_PROMPT_TEMPLATE = """당신은 친절하고 전문적인 공공기관 민원 상담 AI입니다.

## 참고 정보
{context}

## 민원 질문
{question}

## 답변 지침
- 참고 정보가 부족하더라도, 일반적인 지식을 활용하여 상세하게 답변하세요
- 반드시 3문장 이상으로 답변하세요
- 먼저 인사와 공감으로 시작하세요
- 구체적인 해결 방법이나 절차를 안내하세요
- 추가 문의처 안내 시 구체적인 전화번호나 URL을 임의로 생성하지 마세요
- 따뜻하고 친절한 말투를 사용하세요

## 답변"""

ZERO_SHOT_PROMPT_TEMPLATE = """당신은 친절하고 전문적인 공공기관 민원 상담 AI입니다.

## 민원 질문
{question}

## 답변 지침
- 일반적인 지식을 활용하여 상세하게 답변하세요
- 반드시 3문장 이상으로 답변하세요
- 먼저 인사와 공감으로 시작하세요
- 구체적인 해결 방법이나 절차를 안내하세요
- 추가 문의처 안내 시 구체적인 전화번호나 URL을 임의로 생성하지 마세요
- 따뜻하고 친절한 말투를 사용하세요

## 답변"""

print("Prompt templates defined.")
print(f"RAG template length: {len(RAG_PROMPT_TEMPLATE)} chars")
print(f"Zero-shot template length: {len(ZERO_SHOT_PROMPT_TEMPLATE)} chars")

---
## 4. Retrieval + Generation Functions

In [ ]:
def retrieve_context(query: str, top_k: int = 3) -> list:
    """Retrieve top-k similar documents from FAISS index."""
    q_emb = embed_model.encode([query], normalize_embeddings=True,
                                convert_to_numpy=True).astype(np.float32)
    D, I = faiss_index.search(q_emb, top_k)
    
    results = []
    for rank in range(top_k):
        idx = I[0][rank]
        score = float(D[0][rank])
        if 0 <= idx < len(corpus_answers):
            results.append({
                'question': corpus_questions[idx],
                'answer': corpus_answers[idx],
                'score': score,
            })
    return results


def build_rag_prompt(query: str, contexts: list) -> str:
    """Build RAG prompt with retrieved context."""
    context_parts = []
    for i, ctx in enumerate(contexts, 1):
        context_parts.append(f"[{i}] Q: {ctx['question']}\nA: {ctx['answer']}")
    context_str = "\n\n".join(context_parts)
    return RAG_PROMPT_TEMPLATE.format(context=context_str, question=query)


def build_zeroshot_prompt(query: str) -> str:
    """Build zero-shot prompt without context."""
    return ZERO_SHOT_PROMPT_TEMPLATE.format(question=query)


def generate_ollama(prompt: str, model: str = None, max_tokens: int = 512) -> dict:
    """Call Ollama API for text generation."""
    model = model or OLLAMA_MODEL
    if not model:
        return {'response': '[ERROR: No model available]', 'eval_duration': 0}
    
    try:
        r = httpx.post(
            f'{OLLAMA_BASE_URL}/api/generate',
            json={
                'model': model,
                'prompt': prompt,
                'stream': False,
                'options': {
                    'num_predict': max_tokens,
                    'temperature': 0.3,
                },
            },
            timeout=120,
        )
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {'response': f'[ERROR: {e}]', 'eval_duration': 0}


# Quick test
if OLLAMA_MODEL:
    test_result = generate_ollama("안녕하세요, 테스트입니다.", max_tokens=50)
    print(f"Test response: {test_result.get('response', '')[:100]}")
    print(f"Model: {test_result.get('model', 'N/A')}")
else:
    print("Ollama model not available - generation will return error placeholders")

---
## 5. Evaluation

In [ ]:
# --- Run Zero-shot + RAG evaluation ---
N_EVAL_GEN = min(50, len(eval_df))  # Limit for generation (LLM is slow)
print(f"Evaluating {N_EVAL_GEN} samples (Zero-shot + RAG)...")

eval_sample = eval_df.head(N_EVAL_GEN).copy()
zs_answers = []
rag_answers = []
zs_latencies = []
rag_latencies = []
rag_contexts = []

for i, row in eval_sample.iterrows():
    query = row['question']
    
    # Zero-shot
    t0 = time.time()
    zs_prompt = build_zeroshot_prompt(query)
    zs_result = generate_ollama(zs_prompt)
    zs_latency = time.time() - t0
    zs_answers.append(zs_result.get('response', ''))
    zs_latencies.append(zs_latency)
    
    # RAG
    t0 = time.time()
    contexts = retrieve_context(query, top_k=3)
    rag_prompt = build_rag_prompt(query, contexts)
    rag_result = generate_ollama(rag_prompt)
    rag_latency = time.time() - t0
    rag_answers.append(rag_result.get('response', ''))
    rag_latencies.append(rag_latency)
    rag_contexts.append(contexts)
    
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{N_EVAL_GEN}] ZS avg: {np.mean(zs_latencies):.1f}s | RAG avg: {np.mean(rag_latencies):.1f}s")

eval_sample['zs_answer'] = zs_answers
eval_sample['rag_answer'] = rag_answers
eval_sample['zs_latency'] = zs_latencies
eval_sample['rag_latency'] = rag_latencies

print(f"\nDone! ZS avg latency: {np.mean(zs_latencies):.1f}s | RAG avg latency: {np.mean(rag_latencies):.1f}s")

---
## 6. Metrics Computation

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

# --- ROUGE-L ---
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def compute_rouge_l(predictions, references):
    scores = []
    for pred, ref in zip(predictions, references):
        if not pred or not ref:
            scores.append(0.0)
            continue
        result = scorer.score(ref, pred)
        scores.append(result['rougeL'].fmeasure)
    return scores

gt_answers = eval_sample['answer'].tolist()

print("Computing ROUGE-L...")
zs_rouge = compute_rouge_l(zs_answers, gt_answers)
rag_rouge = compute_rouge_l(rag_answers, gt_answers)

print(f"  Zero-shot ROUGE-L: {np.mean(zs_rouge):.4f} (\u00b1{np.std(zs_rouge):.4f})")
print(f"  RAG ROUGE-L:       {np.mean(rag_rouge):.4f} (\u00b1{np.std(rag_rouge):.4f})")

# --- BERTScore ---
print("\nComputing BERTScore (this may take a minute)...")
try:
    P_zs, R_zs, F1_zs = bert_score_fn(
        zs_answers, gt_answers, lang='ko', verbose=False, batch_size=16
    )
    P_rag, R_rag, F1_rag = bert_score_fn(
        rag_answers, gt_answers, lang='ko', verbose=False, batch_size=16
    )
    zs_bertscore = F1_zs.numpy().tolist()
    rag_bertscore = F1_rag.numpy().tolist()
    print(f"  Zero-shot BERTScore F1: {np.mean(zs_bertscore):.4f} (\u00b1{np.std(zs_bertscore):.4f})")
    print(f"  RAG BERTScore F1:       {np.mean(rag_bertscore):.4f} (\u00b1{np.std(rag_bertscore):.4f})")
except Exception as e:
    print(f"  BERTScore failed: {e}")
    zs_bertscore = [0.0] * len(zs_answers)
    rag_bertscore = [0.0] * len(rag_answers)

# Store in DataFrame
eval_sample['zs_rouge'] = zs_rouge
eval_sample['rag_rouge'] = rag_rouge
eval_sample['zs_bertscore'] = zs_bertscore
eval_sample['rag_bertscore'] = rag_bertscore

---
## 7. Visualization

In [ ]:
# --- Grouped bar: Zero-shot vs RAG metrics ---
metrics = {
    'ROUGE-L': [np.mean(zs_rouge), np.mean(rag_rouge)],
    'BERTScore F1': [np.mean(zs_bertscore), np.mean(rag_bertscore)],
}

fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(metrics.keys()),
    y=[v[0] for v in metrics.values()],
    name='Zero-shot',
    marker_color='#ef5350',
    text=[f'{v[0]:.4f}' for v in metrics.values()],
    textposition='outside',
))
fig.add_trace(go.Bar(
    x=list(metrics.keys()),
    y=[v[1] for v in metrics.values()],
    name='RAG',
    marker_color='#42a5f5',
    text=[f'{v[1]:.4f}' for v in metrics.values()],
    textposition='outside',
))

fig.update_layout(
    title='Zero-shot vs RAG: Generation Quality',
    yaxis_title='Score', yaxis_range=[0, 1.1],
    barmode='group',
    width=700, height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show(renderer='iframe')

In [ ]:
# --- Violin plot: score distributions ---
import plotly.graph_objects as go

fig = make_subplots(rows=1, cols=2, subplot_titles=['ROUGE-L Distribution', 'BERTScore F1 Distribution'])

fig.add_trace(go.Violin(y=zs_rouge, name='ZS', line_color='#ef5350',
                         box_visible=True, meanline_visible=True, side='negative'), row=1, col=1)
fig.add_trace(go.Violin(y=rag_rouge, name='RAG', line_color='#42a5f5',
                         box_visible=True, meanline_visible=True, side='positive'), row=1, col=1)

fig.add_trace(go.Violin(y=zs_bertscore, name='ZS', line_color='#ef5350',
                         box_visible=True, meanline_visible=True, side='negative',
                         showlegend=False), row=1, col=2)
fig.add_trace(go.Violin(y=rag_bertscore, name='RAG', line_color='#42a5f5',
                         box_visible=True, meanline_visible=True, side='positive',
                         showlegend=False), row=1, col=2)

fig.update_layout(
    title='Score Distribution: Zero-shot vs RAG',
    width=900, height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1),
)
fig.show(renderer='iframe')

In [ ]:
# --- Latency comparison ---
fig = go.Figure()
fig.add_trace(go.Bar(
    x=['Zero-shot', 'RAG'],
    y=[np.mean(zs_latencies), np.mean(rag_latencies)],
    marker_color=['#ef5350', '#42a5f5'],
    text=[f'{np.mean(zs_latencies):.2f}s', f'{np.mean(rag_latencies):.2f}s'],
    textposition='outside',
    error_y=dict(type='data',
                 array=[np.std(zs_latencies), np.std(rag_latencies)],
                 visible=True),
))

fig.update_layout(
    title='Average Latency: Zero-shot vs RAG',
    yaxis_title='Seconds',
    width=500, height=400,
    showlegend=False,
)
fig.show(renderer='iframe')

---
## 8. Qualitative Examples

In [ ]:
# --- Qualitative comparison: 5 samples ---
html_parts = ['<h3>Qualitative Comparison: Zero-shot vs RAG</h3>']

for i in range(min(5, len(eval_sample))):
    row = eval_sample.iloc[i]
    query = row['question']
    gt = row['answer']
    zs = row['zs_answer']
    rag = row['rag_answer']
    
    html_parts.append(f'''
    <div style="margin:20px 0; padding:15px; border:1px solid #ddd; border-radius:8px;">
        <b>Query {i+1}:</b> {query}<br>
        <b>Ground Truth:</b> <span style="color:#2e7d32;">{gt[:300]}{"..." if len(gt)>300 else ""}</span><br><br>
        <table style="width:100%; border-collapse:collapse;">
            <tr>
                <td style="width:50%; padding:10px; border:1px solid #eee; vertical-align:top;">
                    <b style="color:#ef5350;">Zero-shot</b> (ROUGE-L: {row["zs_rouge"]:.3f})<br>
                    {zs[:300]}{"..." if len(str(zs))>300 else ""}
                </td>
                <td style="width:50%; padding:10px; border:1px solid #eee; vertical-align:top;">
                    <b style="color:#42a5f5;">RAG</b> (ROUGE-L: {row["rag_rouge"]:.3f})<br>
                    {rag[:300]}{"..." if len(str(rag))>300 else ""}
                </td>
            </tr>
        </table>
    </div>
    ''')

display(HTML(''.join(html_parts)))

---
## 9. Save Results

In [ ]:
# --- Save generation_rag_results.json ---
rag_output = {
    'method': 'RAG',
    'llm_model': OLLAMA_MODEL or 'N/A',
    'retriever': best_model_name,
    'top_k': 3,
    'n_eval': len(eval_sample),
    'metrics': {
        'zs_rouge_l': round(float(np.mean(zs_rouge)), 4),
        'rag_rouge_l': round(float(np.mean(rag_rouge)), 4),
        'zs_bertscore': round(float(np.mean(zs_bertscore)), 4),
        'rag_bertscore': round(float(np.mean(rag_bertscore)), 4),
    },
    'latency': {
        'zs_mean_s': round(float(np.mean(zs_latencies)), 2),
        'rag_mean_s': round(float(np.mean(rag_latencies)), 2),
        'zs_std_s': round(float(np.std(zs_latencies)), 2),
        'rag_std_s': round(float(np.std(rag_latencies)), 2),
    },
    'improvement': {
        'rouge_l_delta': round(float(np.mean(rag_rouge) - np.mean(zs_rouge)), 4),
        'bertscore_delta': round(float(np.mean(rag_bertscore) - np.mean(zs_bertscore)), 4),
    },
    # Carry forward retrieval results for pipeline progression
    'retrieval_baseline': {
        'bm25_recall_at_5': bm25_results.get('recall_at_5') if bm25_results else None,
        'sbert_recall_at_5': sbert_results.get('recall_at_5') if sbert_results else None,
    },
}

results_path = os.path.join(RESULTS_DIR, 'generation_rag_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(rag_output, f, ensure_ascii=False, indent=2)

print(f"Results saved: {results_path}")
print(json.dumps(rag_output, ensure_ascii=False, indent=2))

In [ ]:
# --- Base64 download ---
import base64

def create_download_link(filepath, filename=None):
    if filename is None:
        filename = filepath.split('/')[-1]
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / 1024 / 1024
    href = (f'<a href="data:application/octet-stream;base64,{b64}" '
            f'download="{filename}">'
            f'Download: {filename} ({size_mb:.1f} MB)</a>')
    display(HTML(href))

create_download_link(results_path)

---
## 10. Summary

In [ ]:
# --- Pipeline progression chart: BM25 -> SBERT -> RAG ---
stages = []
recall_values = []
colors = []

if bm25_results:
    stages.append('BM25\n(Retrieval)')
    recall_values.append(bm25_results.get('recall_at_5', 0))
    colors.append('#ef5350')

if sbert_results:
    stages.append('SBERT+FAISS\n(Retrieval)')
    recall_values.append(sbert_results.get('recall_at_5', 0))
    colors.append('#42a5f5')

# For RAG, use BERTScore as the "quality" metric
stages.append('RAG\n(Generation)')
recall_values.append(float(np.mean(rag_bertscore)))
colors.append('#66bb6a')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=stages, y=recall_values,
    marker_color=colors,
    text=[f'{v:.4f}' for v in recall_values],
    textposition='outside',
))

fig.update_layout(
    title='Pipeline Progression: Retrieval \u2192 Generation Quality',
    yaxis_title='Score (Recall@5 / BERTScore)',
    yaxis_range=[0, max(recall_values) * 1.3 if recall_values else 1],
    width=700, height=450,
    annotations=[dict(
        text="Retrieval: Recall@5 | Generation: BERTScore F1",
        xref="paper", yref="paper", x=0.5, y=-0.15,
        showarrow=False, font=dict(size=11, color='gray'),
    )],
    margin=dict(b=80),
)
fig.show(renderer='iframe')

In [ ]:
print("=" * 60)
print("     08. RAG Pipeline Evaluation -- Summary")
print("=" * 60)
print()
print(f"  LLM Model:  {OLLAMA_MODEL or 'N/A'}")
print(f"  Retriever:  {best_model_name}")
print(f"  Top-k:      3")
print(f"  Eval size:  {len(eval_sample)}")
print()
print("  Generation Quality:")
print(f"    {'Metric':<20} {'Zero-shot':>10} {'RAG':>10} {'Delta':>10}")
print(f"    {'-'*50}")
print(f"    {'ROUGE-L':<20} {np.mean(zs_rouge):>10.4f} {np.mean(rag_rouge):>10.4f} {np.mean(rag_rouge)-np.mean(zs_rouge):>+10.4f}")
print(f"    {'BERTScore F1':<20} {np.mean(zs_bertscore):>10.4f} {np.mean(rag_bertscore):>10.4f} {np.mean(rag_bertscore)-np.mean(zs_bertscore):>+10.4f}")
print()
print("  Latency:")
print(f"    Zero-shot: {np.mean(zs_latencies):.2f}s (\u00b1{np.std(zs_latencies):.2f})")
print(f"    RAG:       {np.mean(rag_latencies):.2f}s (\u00b1{np.std(rag_latencies):.2f})")
print()
print(f"  Artifacts: {results_path}")
print()
print("  Next -> 09_lora_finetuning")
print("=" * 60)

---
## 11. 하이퍼파라미터 및 평가 설정 근거

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `top_k` | 3 | 검색 문서 3개: 충분한 컨텍스트 제공 + LLM context window 낭비 최소화. top_k=5 이상은 노이즈 문서 유입 가능성 증가 |
| `temperature` | 0.3 | 민원 답변은 **사실 기반 정확성**이 핵심. 높은 temperature는 창의적이지만 hallucination 위험 증가. 0.3은 일관성과 다양성의 균형 |
| `max_tokens` | 512 | 민원 답변의 평균 길이(200~400자) + 여유 확보. 너무 짧으면 불완전한 답변, 너무 길면 반복/장황 |
| `N_EVAL` | 50 | LLM 생성은 쿼리당 수 초 소요 → 전체 시간 제약. 50개 샘플은 ROUGE-L/BERTScore의 평균 추정에 충분 (95% CI ≈ ±0.02) |
| Prompt | RAG vs Zero-shot | 동일 LLM + 동일 프롬프트 구조에서 **컨텍스트 유무만** 변경 → RAG 효과의 순수 측정 |

### 신뢰구간(CI) 참고

N=50 샘플에서 BERTScore 평균의 95% 신뢰구간은 약 `mean ± 1.96 × std / √50` 입니다.
표준편차가 0.10이면 CI ≈ ±0.028, 즉 보고된 평균 차이가 0.03 이상이면 통계적으로 유의미할 가능성이 높습니다.

---
## 12. Cross-Stage 연결: RAG → LoRA Fine-tuning

### RAG의 한계

1. **LLM 자체 능력 의존**: 아무리 좋은 문서를 검색해도, LLM이 한국어 민원 도메인에 익숙하지 않으면 부자연스러운 답변 생성
2. **프롬프트 민감성**: 동일 컨텍스트에서도 프롬프트 표현에 따라 답변 품질 변동
3. **추론 비용**: 매 쿼리마다 검색 + LLM 호출 → 지연 시간 누적
4. **Zero-shot 대비 개선 폭이 기대 이하**인 경우: LLM의 도메인 지식 부족이 병목

### LoRA Fine-tuning (09) 기대 효과

| 개선 포인트 | 메커니즘 |
|------------|---------|
| **도메인 특화** | 민원 QA 쌍으로 직접 학습 → 도메인 특화 어휘/표현 습득 |
| **답변 스타일** | "인사 → 공감 → 안내 → 마무리" 패턴을 학습 데이터에서 자연스럽게 학습 |
| **일관성** | Fine-tuning된 모델은 프롬프트 변동에 덜 민감 |
| **RAG 결합** | Fine-tuned 모델 + RAG = 도메인 지식 + 최신 정보 검색의 시너지 |

**핵심 가설**: RAG baseline 대비 LoRA fine-tuned 모델이 ROUGE-L, BERTScore 모두에서 개선을 보일 것.
특히 **도메인 특화 표현**(인사말, 절차 안내)에서 큰 차이 예상.

In [ ]:
# === Kaggle Dataset 자동 업로드 ===
if IS_KAGGLE:
    UPLOAD_DIR = '/kaggle/working/dataset_upload'
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    # 결과 파일 심볼릭 링크
    for src in [results_path]:
        dst = os.path.join(UPLOAD_DIR, os.path.basename(src))
        if os.path.exists(dst):
            os.remove(dst)
        os.symlink(src, dst)

    meta = {
        "title": "civilcomplaint-rag-pipeline",
        "id": "kukass/civilcomplaint-rag-pipeline",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(UPLOAD_DIR, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    !kaggle datasets create -p {UPLOAD_DIR} --dir-mode zip
    print("✅ Kaggle 데이터셋 업로드 완료: civilcomplaint-rag-pipeline")
else:
    print("ℹ️ 로컬 환경 — Kaggle 업로드 건너뜀")